# 01 — Exploratory Data Analysis (ViHSD)

**Stage 2 goal:** understand the *real* data before any modelling — confirm the
column schema, measure the class (im)balance, look at text-length by class, and
peek at the most frequent character n-grams per class.

We only *explore* here: **no model or vectorizer is fitted into a pipeline or
saved.** All loading goes through `src.data`, which maps the raw `free_text` /
`label_id` columns to a canonical `text` / binary `label` schema and asserts the
schema on load.

In [1]:
import sys
from pathlib import Path

# Notebook runs from notebooks/; add the project root so `import src.*` resolves.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib
matplotlib.use("Agg")  # headless: we save figures to disk, no interactive window
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer

from src.config import ANALYZER, NGRAM_RANGE, FIGURES_DIR
from src.data import load_all

splits = load_all()
{name: df.shape for name, df in splits.items()}

{'train': (24048, 4), 'dev': (2672, 4), 'test': (6680, 4)}

## 1. Schema confirmation

Print the columns and a few rows of each split to confirm the raw schema
(`free_text`, `label_id`) and the two canonical columns added by `src.data`
(`text`, `label`).

In [2]:
for name, df in splits.items():
    print(f"=== {name} ===  columns={list(df.columns)}")
    display(df.head(3))

=== train ===  columns=['free_text', 'label_id', 'text', 'label']


,free_text,label_id,text,label
0,Em được làm fan cứng luôn rồi nè ❤️ reaction q...,0,Em được làm fan cứng luôn rồi nè ❤️ reaction q...,0
1,Đúng là bọn mắt híp lò xo thụt :))) bên việt n...,2,Đúng là bọn mắt híp lò xo thụt :))) bên việt n...,1
2,Đậu Văn Cường giờ giống thằng sida hơn à,0,Đậu Văn Cường giờ giống thằng sida hơn à,0


=== dev ===  columns=['free_text', 'label_id', 'text', 'label']


,free_text,label_id,text,label
0,Coi cười xỉu,0,Coi cười xỉu,0
1,Chi ba vang ngoc dep va tre mai,0,Chi ba vang ngoc dep va tre mai,0
2,"Chần vần một đống, không ai đoán trước được đừ...",0,"Chần vần một đống, không ai đoán trước được đừ...",0


=== test ===  columns=['free_text', 'label_id', 'text', 'label']


,free_text,label_id,text,label
0,Đừng cố biện minh =)))) choi lon,0,Đừng cố biện minh =)))) choi lon,0
1,Haizz. Nthe này thì dân khổ quá,1,Haizz. Nthe này thì dân khổ quá,1
2,the nay ma chi phat gay roi trat tu cong cong ...,0,the nay ma chi phat gay roi trat tu cong cong ...,0


## 2. Binary class distribution

We merge OFFENSIVE(1) + HATE(2) into a single **toxic** class (`label=1`) and keep
CLEAN as `label=0` — a moderation filter only needs BLOCK vs ALLOW. We expect the
toxic class to sit around **~17%** (CLEAN ~82–83%) in every split.

In [3]:
rows = []
for name, df in splits.items():
    counts = df["label"].value_counts().sort_index()
    n = len(df)
    rows.append({
        "split": name,
        "n": n,
        "clean (0)": int(counts.get(0, 0)),
        "toxic (1)": int(counts.get(1, 0)),
        "toxic_ratio": counts.get(1, 0) / n,
        "clean_ratio": counts.get(0, 0) / n,
    })

dist = pd.DataFrame(rows).set_index("split")
dist_display = dist.copy()
dist_display["toxic_ratio"] = dist_display["toxic_ratio"].map("{:.1%}".format)
dist_display["clean_ratio"] = dist_display["clean_ratio"].map("{:.1%}".format)
dist_display

,n,clean (0),toxic (1),toxic_ratio,clean_ratio
split,,,,,
train,24048,19886,4162,17.3%,82.7%
dev,2672,2190,482,18.0%,82.0%
test,6680,5548,1132,16.9%,83.1%


In [4]:
# Sanity check: toxic ratio should be roughly 17% in every split (ViHSD is skewed).
for name, r in dist.iterrows():
    assert 0.13 <= r["toxic_ratio"] <= 0.22, f"{name}: unexpected toxic ratio {r['toxic_ratio']:.3f}"
print("Class balance looks as expected (toxic ~17%, clean ~83%).")

Class balance looks as expected (toxic ~17%, clean ~83%).


## 3. Text length by class

Toxic and clean comments may differ in length. We look at both **character
count** and a **rough whitespace token count** (a crude proxy — Vietnamese needs
real word segmentation, which we intentionally skip at this exploratory stage).

In [5]:
train = splits["train"].copy()
train["n_chars"] = train["text"].astype(str).str.len()
train["n_tokens"] = train["text"].astype(str).str.split().str.len()

# Describe length per binary class (0=clean, 1=toxic) on the train split.
length_summary = train.groupby("label")[["n_chars", "n_tokens"]].describe().T
length_summary

label                      0            1
n_chars  count  19886.000000  4162.000000
         mean      45.272352    68.449784
         std      200.610105    67.938841
         min        1.000000     2.000000
         25%       19.000000    24.000000
         50%       32.000000    46.000000
         75%       51.000000    86.000000
         max    20816.000000   707.000000
n_tokens count  19886.000000  4162.000000
         mean      10.482048    16.407737
         std       20.573122    15.859748
         min        1.000000     1.000000
         25%        5.000000     6.000000
         50%        8.000000    11.000000
         75%       12.000000    21.000000
         max     1701.000000   162.000000

## 4. Top character n-grams per class (exploration only)

We use `CountVectorizer(analyzer="char_wb")` **only to inspect** which character
n-grams dominate each class — `char_wb` is robust to Vietnamese obfuscation and
needs no word segmentation. This vectorizer is **not** part of any pipeline and
is **not** saved; it is thrown away at the end of the cell.

In [6]:
def top_char_ngrams(texts, top_n=20):
    # Throwaway vectorizer: exploration only, never persisted (see markdown above).
    vec = CountVectorizer(analyzer=ANALYZER, ngram_range=NGRAM_RANGE, min_df=5)
    matrix = vec.fit_transform(texts.astype(str))
    freqs = matrix.sum(axis=0).A1  # total count of each n-gram across the corpus
    vocab = vec.get_feature_names_out()
    order = freqs.argsort()[::-1][:top_n]
    return pd.DataFrame({"ngram": vocab[order], "count": freqs[order]})

clean_top = top_char_ngrams(train.loc[train["label"] == 0, "text"])
toxic_top = top_char_ngrams(train.loc[train["label"] == 1, "text"])

pd.concat(
    {"CLEAN (label=0)": clean_top, "TOXIC (label=1)": toxic_top},
    axis=1,
)

CLEAN (label=0)        TOXIC (label=1)       
             ngram  count           ngram  count
0                t  27257               c  10097
1                c  26844               t   8485
2               ng  24650              ng   8324
3               i   23418               n   7984
4               n   22148              n    7692
5                n  22054              i    7463
6               g   18329              g    6221
7              ng   17872             ng    6076
8               nh  15998               đ   5454
9                đ  14144               l   4636
10               l  12274              ch   4336
11              ch  11966              nh   4288
12              th  11864               m   4266
13              th  11630              ch   3823
14               m  10911              th   3776
15              ch  10438              th   3720
16               h  10138              o    3658
17              y   10049               b   3550
18              h   10026              t    3240
19              c    9784              y    3201

## 5. Save the class-distribution figure

Persist a grouped bar chart of clean vs toxic counts per split to
`reports/figures/class_dist.png` (path from `config.FIGURES_DIR`) for the report.

In [7]:
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

ax = dist[["clean (0)", "toxic (1)"]].plot(
    kind="bar", figsize=(7, 4.5), color=["#4C78A8", "#E45756"], edgecolor="black"
)
ax.set_title("ViHSD binary class distribution per split")
ax.set_xlabel("split")
ax.set_ylabel("number of comments")
ax.tick_params(axis="x", rotation=0)
for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=2, fontsize=8)
plt.tight_layout()

out_path = FIGURES_DIR / "class_dist.png"
plt.savefig(out_path, dpi=120)
plt.close()
print(f"Saved figure -> {out_path}  (exists={out_path.exists()})")

Saved figure -> C:\Users\MYPC\OneDrive\Desktop\MAI - Naive Bayes\toxic-comment-filter\reports\figures\class_dist.png  (exists=True)
